# Bacpipe Tutorial

This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---

### Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Loading Audio Files](#2-loading-audio-files)
3. [Ground Truth Labels](#3-ground-truth-labels)
4. [Generating Embeddings — Single File (In-Memory)](#4-generating-embeddings--single-file-in-memory)
5. [Generating Embeddings — Full Directory (Save to Disk)](#5-generating-embeddings--full-directory-save-to-disk)
6. [High-Level Workflow — generate_embeddings](#6-high-level-workflow--generate_embeddings)
7. [Full Pipeline — Single Model](#7-full-pipeline--single-model)
8. [Full Pipeline — Multiple Models](#8-full-pipeline--multiple-models)
9. [End-to-End: bacpipe.play](#9-end-to-end-bacpipeplay)
10. [Benchmarking Classifier Performance](#10-benchmarking-classifier-performance)

---
## 0. Imports & Working Directory

Set the working directory to the repo root and import the display utility.

In [1]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os

# Replace this with the path to the bacpipe repo root on your machine
os.chdir('../..')

---
## 1. Setup & Configuration

Import `bacpipe` and inspect the current configuration and settings. You can also list all available API endpoints, supported models, and the embedding dimensions for each model.

In [2]:
import bacpipe

print('Config:')
display(bacpipe.config)

print('Settings:')
display(bacpipe.settings)

Config:


namespace(audio_dir='bacpipe/tests/test_data',
          overwrite=False,
          dashboard=True,
          models=['birdnet', 'perch_bird'],
          already_computed=False,
          dim_reduction_model='umap',
          evaluation_task=[])

Settings:


namespace(main_results_dir='bacpipe_results',
          embed_parent_dir='embeddings',
          dim_reduc_parent_dir='dim_reduced_embeddings',
          evaluations_dir='evaluations',
          model_base_path='bacpipe/model_checkpoints',
          device='cpu',
          global_batch_size=8,
          audio_suffixes=['.wav',
                          '.WAV',
                          '.aif',
                          '.mp3',
                          '.MP3',
                          '.flac',
                          '.ogg'],
          padding='wrap',
          avoid_pipelined_gpu_inference=False,
          nr_parallel_workers=False,
          rm_embedding_on_keyboard_interrupt=True,
          label_column='species',
          annotations_filename='annotations.csv',
          only_embed_annotations=False,
          default_label_keys=['time_of_day',
                              'day_of_year',
                              'continuous_timestamp',
                              'paren

In [3]:
# All available bacpipe API endpoints
bacpipe.__all__

[<function bacpipe.core.workflows.play(bool_save_logs=False, **kwargs)>,
 <function bacpipe.core.workflows.run_pipeline_for_single_model(model_name, audio_dir, dim_reduction_model='None', check_if_already_processed=True, check_if_already_dim_reduced=True, overwrite=False, testing=False, **kwargs)>,
 <function bacpipe.core.workflows.run_pipeline_for_models(models, audio_dir, dim_reduction_model, **kwargs)>,
 <function bacpipe.core.workflows.generate_embeddings(avoid_pipelined_gpu_inference=False, **kwargs)>,
 bacpipe.core.experiment_manager.Loader,
 bacpipe.model_pipelines.runner.Embedder,
 <function bacpipe.core.experiment_manager.Loader.get_audio_files(audio_dir, audio_suffixes=['.wav', '.WAV', '.aif', '.mp3', '.MP3', '.flac', '.ogg'], return_as='pathlib.Path')>,
 bacpipe.embedding_evaluation.label_embeddings.DefaultLabels,
 <function bacpipe.embedding_evaluation.label_embeddings.create_default_labels(audio_dir=None, model=None, paths=None, overwrite=True, **kwargs)>,
 <function bacpi

In [4]:
# All supported models
bacpipe.supported_models

['audiomae',
 'audioprotopnet',
 'avesecho_passt',
 'aves_especies',
 'bat',
 'beats',
 'birdaves_especies',
 'biolingual',
 'birdnet',
 'birdmae',
 'convnext_birdset',
 'hbdet',
 'insect66',
 'insect459',
 'mix2',
 'naturebeats',
 'perch_bird',
 'perch_v2',
 'protoclr',
 'rcl_fs_bsed',
 'surfperch',
 'google_whale',
 'vggish']

In [5]:
# Embedding dimensions per model
bacpipe.EMBEDDING_DIMENSIONS

{'audiomae': 768,
 'audioprotopnet': 1024,
 'avesecho_passt': 768,
 'aves_especies': 768,
 'bat': 64,
 'beats': 768,
 'birdaves_especies': 1024,
 'biolingual': 512,
 'birdnet': 1024,
 'birdmae': 1280,
 'convnext_birdset': 1024,
 'hbdet': 2048,
 'insect66': 1280,
 'insect459': 1280,
 'mix2': 960,
 'naturebeats': 768,
 'perch_bird': 1280,
 'perch_v2': 1536,
 'protoclr': 384,
 'rcl_fs_bsed': 2048,
 'surfperch': 1280,
 'google_whale': 1280,
 'vggish': 128}

---
## 2. Loading Audio Files

Retrieve all audio files from a directory as a list of strings. 

In [6]:
audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_as='str'
)
audio_files

['bacpipe/tests/test_data/audio/FewShot/CHE_01_20190101_163410.wav',
 'bacpipe/tests/test_data/audio/FewShot/CHE_02_20190101_183410.wav',
 'bacpipe/tests/test_data/audio/FewShot/CHE_03_20190201_163410.wav',
 'bacpipe/tests/test_data/audio/FewShot/CHE_04_20190203_175410.wav',
 'bacpipe/tests/test_data/audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
 'bacpipe/tests/test_data/audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
 'bacpipe/tests/test_data/audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV']

### 2.1 Extract datetime information from filenames

You can also extract datetime information from filenames — useful for datasets where recording time is encoded in the filename.

In [7]:
# Extract datetime information from audio filenames
dt_files = [bacpipe.get_dt_filename(file) for file in audio_files]
dt_files

[datetime.datetime(2019, 1, 1, 16, 34, 10),
 datetime.datetime(2019, 1, 1, 18, 34, 10),
 datetime.datetime(2019, 2, 1, 16, 34, 10),
 datetime.datetime(2019, 2, 3, 17, 54, 10),
 datetime.datetime(2025, 5, 3, 3, 13),
 datetime.datetime(2025, 5, 3, 3, 14),
 datetime.datetime(2025, 5, 3, 3, 15)]

---
## 3. Ground Truth Labels

Load multi-label ground truth annotations and align them to the model's timestamps. Each row in the resulting array corresponds to the same time window as the model's predictions, making it ready for evaluation.

You can also generate a set of default labels for a model/dataset combination, which is useful when no annotations are available.

In [8]:
# Load multi-label ground truth aligned to BirdNET's 3-second time bins
gt = bacpipe.ground_truth_by_model(
    'birdnet',
    audio_dir='bacpipe/tests/test_data',
    annotations_filename='annotations.csv',
    single_label=False
)
gt

{'label:species': array([[ 0., -1.],
        [ 0., -1.],
        [ 0., -1.],
        [ 1., -1.],
        [ 1., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 2., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 3., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 5., -1.],
        [ 5., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
        [ 4., -1.],
   

In [9]:
# Generate default labels for a model/dataset combination
dl = bacpipe.create_default_labels(
    'bacpipe/tests/test_data',
    'birdnet'
)
dl

{'time_of_day': ['16-34-10',
  '16-34-13',
  '16-34-16',
  '16-34-19',
  '16-34-22',
  '16-34-25',
  '16-34-28',
  '16-34-31',
  '16-34-34',
  '16-34-37',
  '16-34-40',
  '16-34-43',
  '16-34-46',
  '16-34-49',
  '16-34-52',
  '16-34-55',
  '16-34-58',
  '16-35-01',
  '16-35-04',
  '16-35-07',
  '16-35-10',
  '16-35-13',
  '18-34-10',
  '18-34-13',
  '18-34-16',
  '18-34-19',
  '16-34-10',
  '16-34-13',
  '16-34-16',
  '17-54-10',
  '17-54-13',
  '17-54-16',
  '17-54-19',
  '03-13-00',
  '03-13-03',
  '03-13-06',
  '03-13-09',
  '03-13-12',
  '03-13-15',
  '03-13-18',
  '03-13-21',
  '03-13-24',
  '03-13-27',
  '03-14-00',
  '03-14-03',
  '03-14-06',
  '03-14-09',
  '03-14-12',
  '03-14-15',
  '03-14-18',
  '03-14-21',
  '03-14-24',
  '03-14-27',
  '03-15-00',
  '03-15-03',
  '03-15-06',
  '03-15-09',
  '03-15-12',
  '03-15-15',
  '03-15-18',
  '03-15-21',
  '03-15-24',
  '03-15-27'],
 'day_of_year': ['2000-01-01',
  '2000-01-01',
  '2000-01-01',
  '2000-01-01',
  '2000-01-01',
  '2000

---
## 4. Generating Embeddings — Single File (In-Memory)

The `Loader` and `Embedder` classes are the core building blocks of bacpipe. You can instantiate them directly to generate embeddings for a single file without saving anything to disk. This is useful for quick experimentation or when you want to work with embeddings directly in memory.

In [10]:
# Create a Loader without folder structure — nothing is written to disk
loader_obj = bacpipe.Loader('bacpipe/tests/test_data')
embed_obj = bacpipe.Embedder('birdnet', loader_obj)

# Generate embeddings for the first audio file
embeds = embed_obj.get_embeddings_from_model(loader_obj.files[0])

print('Since embeddings were not saved, .embeddings() returns empty:')
display(loader_obj.embeddings())

print('The embeddings are still accessible via the declared variable:')
display(embeds)

print('Classifier predictions are accessible through the embedder object:')
display(embed_obj.classifier.predictions)

No model_name is passed, therefore no directory structure will be created.
2026-04-10 10:16:47.653267: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 10:16:47.685231: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using device='cpu'
2026-04-10 10:16:50.575887: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-04-10 10:16:50.575908: I external/local_xla/xla/stream_executor/cu

Since embeddings were not saved, .embeddings() returns empty:


{}

The embeddings are still accessible via the declared variable:


array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109774, 0.        , ..., 0.44500875, 0.30178487,
        1.4982901 ],
       [0.        , 0.04689773, 0.        , ..., 0.        , 0.        ,
        0.3570976 ],
       ...,
       [0.        , 0.6240226 , 0.19019024, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.28852096, 0.27240792, ..., 0.        , 0.        ,
        0.5682918 ],
       [0.        , 0.3114516 , 0.18153691, ..., 0.        , 0.        ,
        0.45023656]], dtype=float32)

Classifier predictions are accessible through the embedder object:


tensor([[7.2337e-06, 1.2321e-06, 1.8781e-05,  ..., 9.8536e-06, 4.5944e-06,
         8.6071e-06],
        [2.6510e-06, 2.3206e-07, 8.3774e-06,  ..., 1.9020e-06, 1.8968e-06,
         4.7985e-07],
        [7.6319e-06, 1.2511e-05, 3.5126e-05,  ..., 5.9355e-05, 2.0990e-04,
         5.9185e-06],
        ...,
        [1.2915e-06, 1.7141e-07, 2.1810e-06,  ..., 1.5112e-04, 6.5344e-05,
         1.2531e-05],
        [1.5508e-05, 9.8838e-07, 7.9295e-07,  ..., 3.2521e-05, 2.3045e-05,
         1.1301e-05],
        [2.5794e-06, 3.8611e-06, 4.1147e-06,  ..., 4.6988e-05, 2.5489e-05,
         4.5167e-05]])

---
## 5. Generating Embeddings — Full Directory (Save to Disk)

To process all audio files in a directory and persist the results, pass `use_folder_structure=True` to the `Loader`. Bacpipe will create a timestamped output directory, run inference using multithreading, and save embeddings, metadata, and classifier predictions. Subsequent runs will detect the saved files and skip recomputation automatically.

In [ ]:
MODEL_NAME = 'birdnet'

loader_obj = bacpipe.Loader(
    'bacpipe/tests/test_data',
    MODEL_NAME,
    use_folder_structure=True
)
embed_obj = bacpipe.Embedder(MODEL_NAME, loader_obj)

# Process all files using multithreading
embed_obj.run_inference_pipeline_using_multithreading(
    
)

print('Embeddings (array):')
display(loader_obj.embeddings(return_type='array'))

print('Metadata dict:')
display(loader_obj.metadata_dict)

print('Predictions (array):')
display(loader_obj.predictions(return_type='array'))

print('Predictions (dataframe):')
display(loader_obj.predictions(return_type='dataframe'))


The directory bacpipe_results/test_data/embeddings/2026-04-10_10-24___birdnet-test_data is not empty. It seems like a previous run failed, bacpipe is comparing what files were already created and will then continue where it left off.If you interrupted the run on purpose and want to start from the beginning, please cancel using Ctrl + C and then remove the folder bacpipe_results/test_data/embeddings/2026-04-10_10-24___birdnet-test_data manually.

Using device='cpu'
Skipping model.eval() because model is from tensorflow.


Embeddings (array):


array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109774, 0.        , ..., 0.44500875, 0.30178487,
        1.4982901 ],
       [0.        , 0.04689773, 0.        , ..., 0.        , 0.        ,
        0.3570976 ],
       ...,
       [0.        , 0.36581087, 0.22898036, ..., 1.8080926 , 0.31529734,
        1.3595358 ],
       [0.        , 0.02800163, 0.02318063, ..., 0.48662066, 0.        ,
        0.6104872 ],
       [0.        , 0.1318497 , 0.32147038, ..., 1.2065995 , 0.6049798 ,
        0.73578507]], dtype=float32)

Metadata dict:


{'model_name': 'birdnet',
 'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/test_data/embeddings/2026-04-10_10-24___birdnet-test_data',
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [66.0, 12.0, 9.0, 12.0, 30.0, 30.0, 30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'segment_length (samples)': 144000,
 'sample_rate (Hz)': 48000,
 'nr_embeds_total': 63,
 'total_dataset_length (s)': 189.0}

Predictions (array):


(array([[0.        , 0.        , 0.        , 0.        ],
        [0.6252951 , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.55551088, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.73798287, 0.        , 0.        ],
        [0.88011229, 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.   

Predictions (dataframe):


,Unnamed: 0,audiofilename,start,end,species_richness,Eurasian Blackbird,Common Cuckoo,Dunnock,Short-toed Treecreeper
0,1,audio/FewShot/CHE_01_20190101_163410.wav,3.0,6.0,1,0.000000,0.000000,0.000000,0.625295
1,3,audio/FewShot/CHE_01_20190101_163410.wav,9.0,12.0,1,0.000000,0.000000,0.000000,0.555511
2,11,audio/FewShot/CHE_01_20190101_163410.wav,33.0,36.0,1,0.000000,0.000000,0.737983,0.000000
3,12,audio/FewShot/CHE_01_20190101_163410.wav,36.0,39.0,1,0.000000,0.000000,0.000000,0.880112
4,22,audio/FewShot/CHE_02_20190101_183410.wav,0.0,3.0,1,0.000000,0.604620,0.000000,0.000000
5,23,audio/FewShot/CHE_02_20190101_183410.wav,3.0,6.0,1,0.000000,0.817276,0.000000,0.000000
6,24,audio/FewShot/CHE_02_20190101_183410.wav,6.0,9.0,1,0.000000,0.606057,0.000000,0.000000
7,25,audio/FewShot/CHE_02_20190101_183410.wav,9.0,12.0,1,0.000000,0.703758,0.000000,0.000000
8,26,audio/FewShot/CHE_03_20190201_163410.wav,0.0,3.0,1,0.000000,0.510690,0.000000,0.000000
9,27,audio/FewShot/CHE_03_20190201_163410.wav,3.0,6.0,1,0.000000,0.584851,0.000000,0.000000


---
## 6. High-Level Workflow — `generate_embeddings`

`bacpipe.generate_embeddings` wraps the `Loader`/`Embedder` pattern into a single convenient call. It handles folder creation, embedding generation, and saving automatically.

In [ ]:
# 
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data'
)

display(loader_obj.metadata_dict)
display(loader_obj.embeddings(return_type='array'))




###### Generating embeddings using BIRDNET ######


### Embeddings already exist. Using embeddings in bacpipe_results/test_data/embeddings/2026-04-10_10-24___birdnet-test_data ###


{'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/test_data/embeddings/2026-04-10_10-24___birdnet-test_data',
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [66.0, 12.0, 9.0, 12.0, 30.0, 30.0, 30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'model_name': 'birdnet',
 'nr_embeds_total': 63,
 'sample_rate (Hz)': 48000,
 'segment_length (samples)': 144000,
 'total_dataset_length (s)': 189.0}

array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109774, 0.        , ..., 0.44500875, 0.30178487,
        1.4982901 ],
       [0.        , 0.04689773, 0.        , ..., 0.        , 0.        ,
        0.3570976 ],
       ...,
       [0.        , 0.36581087, 0.22898036, ..., 1.8080926 , 0.31529734,
        1.3595358 ],
       [0.        , 0.02800163, 0.02318063, ..., 0.48662066, 0.        ,
        0.6104872 ],
       [0.        , 0.1318497 , 0.32147038, ..., 1.2065995 , 0.6049798 ,
        0.73578507]], dtype=float32)

### 6.1 Load and concatenate audio files yourself, then pass the result to bacpipe

You can also load and preprocess audio yourself and pass it directly to an `Embedder` — useful when you need custom loading logic or want to process audio that isn't stored on disk.

In [20]:
# Alternatively: load and concatenate audio yourself, then pass it directly to an Embedder
import librosa as lb
import numpy as np

audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_as='str'
)

audio = []
for file in audio_files:
    aud, sr = lb.load(file)
    audio.extend(aud)
audio = np.array(audio)

# check if the model exists, if not, download it
bacpipe.ensure_models_exist(bacpipe.settings.model_base_path, ['naturebeats'])
# embed
embed_obj = bacpipe.Embedder('naturebeats')
embeds = embed_obj.embeddings_using_multithreading(audio)
embeds

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

naturebeats checkpoint does not exists. Downloading the model from https://huggingface.co/datasets/vskode/bacpipe_models/blob/main/naturebeats/naturebeats.tar.xz



naturebeats/naturebeats.tar.xz:   0%|          | 0.00/724M [00:00<?, ?B/s]

beats checkpoint does not exists. Downloading the model from https://huggingface.co/datasets/vskode/bacpipe_models/blob/main/beats/beats.tar.xz



beats/beats.tar.xz:   0%|          | 0.00/726M [00:00<?, ?B/s]

Using device='cpu'
BEATs Config: {'input_patch_size': 16, 'embed_dim': 512, 'conv_bias': False, 'encoder_layers': 12, 'encoder_embed_dim': 768, 'encoder_ffn_embed_dim': 3072, 'encoder_attention_heads': 12, 'activation_fn': 'gelu', 'layer_wise_gradient_decay_ratio': 0.6, 'layer_norm_first': False, 'deep_norm': True, 'dropout': 0.0, 'attention_dropout': 0.0, 'activation_dropout': 0.0, 'encoder_layerdrop': 0.05, 'dropout_input': 0.0, 'conv_pos': 128, 'conv_pos_groups': 16, 'relative_position_embedding': True, 'num_buckets': 320, 'max_distance': 800, 'gru_rel_pos': True, 'finetuned_model': True, 'predictor_dropout': 0.0, 'predictor_class': 527}
/home/haupert/miniconda3/envs/bacpipe/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Skipping model.eval() because model is from tensorflow.
/media/haupert/data/mes_projets/_packa

[array([-4.83296365e-02, -2.11147219e-01, -6.51899874e-02,  1.54395089e-01,
        -1.93823740e-01,  1.20961271e-01, -5.23437679e-01,  6.69806376e-02,
        -1.82811931e-01,  5.84075928e-01,  1.99450720e-02,  1.05686620e-01,
         2.06550553e-01,  7.54849792e-01, -5.18227339e-01,  2.84695029e-01,
        -3.19758117e-01, -1.65323615e-01,  5.33026457e-01, -5.15896916e-01,
        -2.02797249e-01, -3.62540871e-01, -1.18120782e-01,  1.44641951e-01,
         2.30196819e-01,  9.11036208e-02,  2.44898751e-01, -1.91884965e-01,
         7.18611702e-02,  8.88762623e-03,  2.03647837e-02, -5.12050986e-01,
        -1.74788117e-01,  8.41685310e-02, -1.79470077e-01,  1.34762362e-01,
        -5.40641248e-02, -1.13048993e-01,  1.38467386e-01,  1.67244911e-01,
         1.49834678e-01,  4.62805510e-01, -3.70641023e-01, -1.43073380e-01,
        -2.13490754e-01,  1.24341380e-02,  1.69065017e-02, -1.74506068e-01,
        -1.19819604e-01, -3.35741863e-02,  8.73093382e-02, -2.91836768e-01,
        -1.4

---
## 7. Full Pipeline — Single Model

`run_pipeline_for_single_model` runs the complete bacpipe pipeline for one model: embedding generation, classifier inference, optional dimensionality reduction, and visualisation. It returns a `Loader` object with all results accessible.

In [ ]:
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)
loader_obj

---
## 8. Full Pipeline — Multiple Models

`run_pipeline_for_models` runs the full pipeline across a list of models in one call, returning a dictionary of `Loader` objects keyed by model name. This makes it straightforward to compare embeddings and predictions across models on the same dataset.

In [ ]:
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'naturebeats'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)

display(loader_dictionary['birdnet'].metadata_dict)
display(loader_dictionary['naturebeats'].embeddings())

---
## 9. End-to-End: `bacpipe.play`

`bacpipe.play` is the highest-level entry point. It runs the complete pipeline — embeddings, classification, dimensionality reduction, evaluation, and an interactive dashboard — in a single call. Ideal for a first exploration of a new dataset across multiple models.

In [ ]:
bacpipe.play(
    models=['birdnet', 'perch_bird', 'naturebeats'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)

---
## 10. Benchmarking Classifier Performance

`bacpipe.benchmark` evaluates a model's pretrained classifier against ground truth annotations. It aligns predictions and ground truth to the same timestamps, handles label mismatches with a regex fallback for hyphen/spacing differences, and returns a per-species `sklearn` classification report with precision, recall, and F1.

If predictions have already been generated for this dataset, the function loads them from disk and runs very quickly.

In [ ]:
results = bacpipe.benchmark(
    'birdnet',
    'bacpipe/tests/test_data',
    annotations_file='annotations.csv'
)
display(results)